# 🏥 MedScript AI — Model Training Notebook

**NeuroScript Clinical Engine** — Train the full pipeline:
1. **Donut Vision Encoder** (Swin Transformer) → Extract features from prescription images
2. **BiLSTM-CTC Decoder** → Decode features to character sequences
3. **BiomedBERT NER** → Extract structured medical entities

### Prerequisites
- Google Colab with **T4 GPU** (free tier works)
- This notebook is self-contained — it clones the repo and installs deps

---

## 0. Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Mount Google Drive (to save checkpoints persistently)
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory in Drive
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/medscript-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
# Clone the repository
# Replace with your actual repo URL
!git clone https://github.com/nvk170405/medscriptai.git /content/medscript-ai 2>/dev/null || echo "Repo already exists"
%cd /content/medscript-ai

In [ ]:
# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q pytorch-lightning>=2.2.0 transformers>=4.40.0 tokenizers
!pip install -q albumentations>=1.4.0 opencv-python-headless
!pip install -q rapidfuzz numpy Pillow pyyaml fonttools
!pip install -q structlog

# Install the package in editable mode
!pip install -q -e . --no-deps

print("✅ All dependencies installed")

## 1. Generate Synthetic Training Data

Since we don't have a real labeled dataset of handwritten prescriptions, we generate synthetic data using random fonts and medical text templates.

In [ ]:
import yaml
from pathlib import Path

# Load configs
with open('configs/model_config.yaml') as f:
    model_config = yaml.safe_load(f)

with open('configs/training_config.yaml') as f:
    training_config = yaml.safe_load(f)

print("Model config:")
print(f"  Encoder: {model_config['donut']['pretrained_model']}")
print(f"  Input size: {model_config['donut']['input_size']}")
print(f"  BiLSTM hidden: {model_config['bilstm']['hidden_size']}")
print(f"  Vocab size: {model_config['vocabulary']['vocab_size']}")
print(f"  NER model: {model_config['medical_bert']['pretrained_model']}")
print(f"\nTraining config:")
print(f"  Batch size: {training_config['training']['batch_size']}")
print(f"  Grad accum: {training_config['training']['gradient_accumulation_steps']}")
print(f"  Effective batch: {training_config['training']['batch_size'] * training_config['training']['gradient_accumulation_steps']}")
print(f"  Max epochs: {training_config['training']['max_epochs']}")
print(f"  Precision: {training_config['training']['precision']}")

In [ ]:
import sys
sys.path.append('./src')
from medscript.data.synthetic import SyntheticPrescriptionGenerator

# Generate synthetic data
NUM_TRAIN_SAMPLES = 2000   # Increase for better results (5000+ recommended)
NUM_VAL_SAMPLES = 200
OUTPUT_DIR = Path('data/synthetic')

generator = SyntheticPrescriptionGenerator(
    output_dir=str(OUTPUT_DIR),
    image_height=960,
    image_width=1280,
    seed=42,
)

print(f"Generating {NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES} synthetic prescriptions...")
generator.generate_batch(count=NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES)
print(f"✅ Generated {NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES} samples in {OUTPUT_DIR}")

In [ ]:
# Visualize a few samples
import json
from PIL import Image
import matplotlib.pyplot as plt

with open(OUTPUT_DIR / 'annotations.json') as f:
    annotations = json.load(f)

print(f"Total annotations: {len(annotations)}")
print(f"Sample annotation keys: {list(annotations[0].keys())}")

# Show first 4 samples
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    if i < len(annotations):
        ann = annotations[i]
        img_path = OUTPUT_DIR / ann['image_path']
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray')
            ax.set_title(ann['full_text'][:60] + '...', fontsize=8)
        ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Build Vocabulary & DataModule

In [ ]:
# Build character vocabulary from config
charset = model_config['vocabulary']['charset']

# Index 0 = CTC blank token
char_to_idx = {char: idx + 1 for idx, char in enumerate(charset)}
idx_to_char = {idx + 1: char for idx, char in enumerate(charset)}
idx_to_char[0] = ''  # blank token maps to empty string

VOCAB_SIZE = len(charset) + 1  # +1 for blank

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Characters: {charset[:50]}...")
print(f"Sample mapping: 'A' -> {char_to_idx.get('A', '?')}")

In [ ]:
from medscript.data.datamodule import MedScriptDataModule

# Create DataModule
datamodule = MedScriptDataModule(
    data_dirs=['data/synthetic'],
    batch_size=training_config['training']['batch_size'],
    num_workers=2,       # Colab has limited CPU cores
    pin_memory=True,
    image_height=960,
    image_width=1280,
    max_text_length=model_config['donut']['max_length'],
    train_split=0.85,
    val_split=0.15,
    test_split=0.0,
    augmentation_level='medium',
    curriculum_enabled=True,
    seed=42,
)

# Setup datasets
datamodule.setup('fit')

print(f"Training samples: {len(datamodule.train_dataset)}")
print(f"Validation samples: {len(datamodule.val_dataset)}")

# Test a batch
train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  images: {batch['images'].shape}")
print(f"  targets: {batch['targets'].shape}")
print(f"  target_lengths: {batch['target_lengths']}")
print(f"  texts: {batch['texts'][:2]}")

## 3. Initialize Model

In [ ]:
from medscript.training.lightning_module import MedScriptLightningModule

# Initialize Lightning module
model = MedScriptLightningModule(
    # Model architecture
    pretrained_donut=model_config['donut']['pretrained_model'],
    encoder_output_dim=model_config['bilstm']['input_dim'],
    bilstm_hidden_size=model_config['bilstm']['hidden_size'],
    bilstm_num_layers=model_config['bilstm']['num_layers'],
    bilstm_dropout=model_config['bilstm']['dropout'],
    vocab_size=VOCAB_SIZE,
    freeze_encoder=False,
    # Training hyperparameters
    learning_rate=training_config['training']['learning_rate'],
    bilstm_learning_rate=training_config['training']['bilstm_learning_rate'],
    weight_decay=training_config['training']['weight_decay'],
    warmup_steps=training_config['training']['warmup_steps'],
    max_epochs=training_config['training']['max_epochs'],
    # Vocab
    idx_to_char=idx_to_char,
    use_pretrained=True,
)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model Summary")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters:    {total_params - trainable_params:,}")

# Estimate VRAM usage
param_size_gb = total_params * 4 / 1e9  # fp32
print(f"  Est. VRAM (fp32):     {param_size_gb:.2f} GB")
print(f"  Est. VRAM (fp16):     {param_size_gb / 2:.2f} GB")

## 4. Train — Stage 1: Encoder + Decoder (CTC Loss)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
    RichProgressBar,
)

# Callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename='medscript-{epoch:02d}-{val_wer:.4f}',
    monitor='val/wer',
    mode='min',
    save_top_k=3,
    save_last=True,
    verbose=True,
)

early_stop_callback = EarlyStopping(
    monitor='val/wer',
    patience=5,
    mode='min',
    min_delta=0.001,
    verbose=True,
)

lr_monitor = LearningRateMonitor(logging_interval='step')

# Trainer
trainer = pl.Trainer(
    max_epochs=training_config['training']['max_epochs'],
    accelerator='gpu',
    devices=1,
    precision=training_config['training']['precision'],
    gradient_clip_val=training_config['training']['gradient_clip_val'],
    accumulate_grad_batches=training_config['training']['gradient_accumulation_steps'],
    val_check_interval=0.5,  # Validate every half epoch
    log_every_n_steps=10,
    callbacks=[checkpoint_callback, early_stop_callback, lr_monitor],
    deterministic=True,
    enable_progress_bar=True,
)

print("🚀 Starting training...")
print(f"   Precision: {training_config['training']['precision']}")
print(f"   Batch: {training_config['training']['batch_size']} x {training_config['training']['gradient_accumulation_steps']} accum = {training_config['training']['batch_size'] * training_config['training']['gradient_accumulation_steps']} effective")
print(f"   Epochs: {training_config['training']['max_epochs']}")
print(f"   Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
# Set training stage (encoder + decoder, NER frozen)
model.model.set_training_stage('encoder_decoder')

# Train!
trainer.fit(model, datamodule=datamodule)

print(f"\n✅ Training complete!")
print(f"   Best model: {checkpoint_callback.best_model_path}")
print(f"   Best WER: {checkpoint_callback.best_model_score:.4f}")

## 5. Evaluate

In [ ]:
# Load best checkpoint
best_model = MedScriptLightningModule.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    idx_to_char=idx_to_char,
)
best_model.eval()
best_model.cuda()

# Run on validation set
val_loader = datamodule.val_dataloader()

all_preds = []
all_refs = []

with torch.no_grad():
    for batch in val_loader:
        images = batch['images'].cuda()
        texts = batch['texts']

        log_probs = best_model(images)
        decoded = best_model.model.decoder.greedy_decode(log_probs)

        for indices in decoded:
            pred_text = ''.join(idx_to_char.get(idx, '') for idx in indices)
            all_preds.append(pred_text)

        all_refs.extend(texts)

# Compute metrics
from medscript.training.metrics import word_error_rate, character_error_rate

wer = word_error_rate(all_preds, all_refs)
cer = character_error_rate(all_preds, all_refs)

print(f"📊 Validation Results")
print(f"   WER: {wer:.4f} ({(1 - wer) * 100:.1f}% word accuracy)")
print(f"   CER: {cer:.4f} ({(1 - cer) * 100:.1f}% char accuracy)")

# Show sample predictions
print(f"\n📝 Sample Predictions (first 5):")
for i in range(min(5, len(all_preds))):
    print(f"  REF:  {all_refs[i][:80]}")
    print(f"  PRED: {all_preds[i][:80]}")
    print()

## 6. Export Best Checkpoint for Deployment

Copy the best checkpoint to the format expected by the FastAPI backend.

In [ ]:
import shutil

# Copy best checkpoint for API deployment
best_ckpt = checkpoint_callback.best_model_path
deploy_ckpt = '/content/medscript-ai/checkpoints/best.ckpt'
os.makedirs(os.path.dirname(deploy_ckpt), exist_ok=True)
shutil.copy2(best_ckpt, deploy_ckpt)

# Also save vocabulary mapping
import json
vocab_path = '/content/medscript-ai/checkpoints/vocab.json'
with open(vocab_path, 'w') as f:
    json.dump({
        'char_to_idx': char_to_idx,
        'idx_to_char': {str(k): v for k, v in idx_to_char.items()},
        'vocab_size': VOCAB_SIZE,
    }, f, indent=2)

# Save to Drive too
shutil.copy2(deploy_ckpt, os.path.join(CHECKPOINT_DIR, 'best.ckpt'))
shutil.copy2(vocab_path, os.path.join(CHECKPOINT_DIR, 'vocab.json'))

print(f"✅ Deployment checkpoint saved:")
print(f"   Local: {deploy_ckpt}")
print(f"   Drive: {CHECKPOINT_DIR}/best.ckpt")
print(f"   Vocab: {vocab_path}")
print(f"\n📦 To deploy: Download best.ckpt and vocab.json to your local")
print(f"   e:\\medscript-ai\\checkpoints\\ directory, then restart the API.")

In [ ]:
# Download files directly from Colab
from google.colab import files

print("Downloading best.ckpt...")
files.download(deploy_ckpt)

print("Downloading vocab.json...")
files.download(vocab_path)